# LLM-as-Judge for Mrigi 23i: Multi-Model Generation Quality

**Mrigi 30a** — Evaluates the outputs from Mrigi 23i (14 models, 100-question Selected MCQs v2 dataset) using GPT-4.1 as a judge.

This is a sibling of Mrigi 30 (which judged 23e's single-model × multi-k × multi-category outputs).
23i's structure is flatter: many models × two methods (no_context, mmr at k=15), so the iteration here is `model × method` instead of `k × category × method`.

**Three Evaluation Tasks (scored 1-10 with reasoning, same as Mrigi 30):**
1. **Context Relevance** — Given a query, how relevant is the retrieved context? (skipped for `no_context`)
2. **Response Correctness** — Given the context, how correct is the model's response?
3. **Response Completeness** — Given the query, how complete is the model's response?

**Input:** `results_23h_checkpoint.json` (always reflects 23i's current state).  
**Judge:** GPT-4.1 via OpenAI API, parallelized (8 worker threads).  
**Output:** per-question JSON scores, aggregated CSV, grouped-bar plots per task + per-method model×task heatmaps.

In [2]:
# Core imports
import os
import json
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
import httpx

# Load environment variables (OPENAI_API_TOKEN, HF_TOKEN)
with open('/home/jupyter/Mrigi/env.sh') as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            key, val = line[len('export '):].split('=', 1)
            os.environ[key] = val.strip('"').strip("'")

openai_key = os.environ.get("OPENAI_API_TOKEN")
if not openai_key:
    raise ValueError("OPENAI_API_TOKEN not found in /home/jupyter/Mrigi/env.sh")

client = OpenAI(api_key=openai_key, http_client=httpx.Client())
print(f"✓ OpenAI client initialized")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ OpenAI client initialized
✓ Timestamp: 2026-05-27 12:16:56


## Load 23i results from `results_23h_checkpoint.json`

The checkpoint is the canonical source of truth — 23i writes to it after every model completes.

In [3]:
# Load the 23i checkpoint
RESULTS_FILE = 'results_23h_checkpoint.json'

if not os.path.exists(RESULTS_FILE):
    # Fallback: list candidate files so user can adjust
    candidates = sorted(glob.glob('results_23h_*.json'))
    print("Available 23h/23i result files:")
    for c in candidates:
        print(f"  {c}  ({os.path.getsize(c)/(1024*1024):.1f} MB)")
    raise FileNotFoundError(f"{RESULTS_FILE!r} not found. Run 23i first or point RESULTS_FILE at one of the above.")

with open(RESULTS_FILE) as f:
    all_results = json.load(f)

print(f"Loaded: {RESULTS_FILE} ({os.path.getsize(RESULTS_FILE)/(1024*1024):.2f} MB)")
print(f"Models in file: {len(all_results)}")
for model_name, model_data in all_results.items():
    if not isinstance(model_data, dict):
        print(f"  {model_name}: (unexpected shape: {type(model_data).__name__})")
        continue
    if 'load_error' in model_data:
        print(f"  {model_name}: LOAD ERROR ({str(model_data['load_error'])[:60]})")
        continue
    methods = [m for m in model_data.keys() if m in ('no_context', 'mmr')]
    parts = []
    for m in methods:
        r = model_data.get(m, {})
        n = len(r.get('detailed_results', []))
        acc = r.get('accuracy', None)
        parts.append(f"{m}={n}q" + (f" acc={acc:.1f}%" if isinstance(acc, (int, float)) else ""))
    print(f"  {model_name:30s}  " + "  ".join(parts))

Loaded: results_23h_checkpoint.json (10.19 MB)
Models in file: 14
  Llama-3-8B-Instruct             no_context=100q acc=97.0%  mmr=100q acc=99.0%
  DAPT_LR1e5                      no_context=100q acc=97.0%  mmr=100q acc=99.0%
  DAPT_LR1e5_COT: LOAD ERROR (We couldn't connect to 'https://huggingface.co' to load the )
  synv2V2_step80                  no_context=100q acc=98.0%  mmr=100q acc=99.0%
  synv2V2_step80_COT: LOAD ERROR (We couldn't connect to 'https://huggingface.co' to load the )
  synv2V2_final                   no_context=100q acc=98.0%  mmr=100q acc=99.0%
  synv2V2_final_COT: LOAD ERROR (We couldn't connect to 'https://huggingface.co' to load the )
  synv2_base_step80               no_context=100q acc=0.0%  mmr=100q acc=0.0%
  synv2_base_step80_COT: LOAD ERROR (We couldn't connect to 'https://huggingface.co' to load the )
  synv2_base_final                no_context=100q acc=0.0%  mmr=100q acc=0.0%
  synv2_base_final_COT: LOAD ERROR (We couldn't connect to 'https://huggingf

In [4]:
# Determine which models are judgeable: skip load_errors, require both methods present with detailed_results
MODELS_TO_JUDGE = []
SKIPPED = []
for model_name, model_data in all_results.items():
    if not isinstance(model_data, dict) or 'load_error' in model_data:
        SKIPPED.append((model_name, 'load_error or unexpected shape'))
        continue
    has_nc = 'detailed_results' in model_data.get('no_context', {}) and len(model_data['no_context']['detailed_results']) > 0
    has_mmr = 'detailed_results' in model_data.get('mmr', {}) and len(model_data['mmr']['detailed_results']) > 0
    if has_nc and has_mmr:
        MODELS_TO_JUDGE.append(model_name)
    else:
        SKIPPED.append((model_name, f"incomplete: no_context={has_nc}, mmr={has_mmr}"))

METHODS = ['no_context', 'mmr']
print(f"✓ Will judge {len(MODELS_TO_JUDGE)} models × {len(METHODS)} methods")
for m in MODELS_TO_JUDGE:
    print(f"    ✓ {m}")
if SKIPPED:
    print(f"
⚠ Skipping {len(SKIPPED)} models:")
    for name, why in SKIPPED:
        print(f"    ✗ {name}  ({why})")

SyntaxError: EOL while scanning string literal (1259657112.py, line 20)

## GPT-4.1 Judge Prompts

Same three rubrics as Mrigi 30 (context-relevance, response-correctness, response-completeness).

In [ ]:
JUDGE_SYSTEM_PROMPT = """You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system 
focused on zeolite synthesis, catalysis, and environmental applications. 
You will evaluate the quality of retrieved contexts and generated responses.
Always respond in valid JSON format."""

CONTEXT_RELEVANCE_PROMPT = """Given the following query and retrieved context, rate how relevant the context is for answering the query.

**Scoring Rubric (1-10):**
- 1-2: Completely irrelevant — context has no connection to the query topic
- 3-4: Mostly irrelevant — context touches on the general domain but doesn't address the specific question
- 5-6: Partially relevant — context contains some useful information but misses key aspects
- 7-8: Mostly relevant — context addresses the main topic and provides useful information for answering
- 9-10: Highly relevant — context directly addresses the query with specific, useful information

**Query:**
{query}

**Retrieved Context:**
{context}

Respond with ONLY a JSON object in this exact format:
{{"score": <integer 1-10>, "reasoning": "<one to two sentences explaining your score>"}}"""

RESPONSE_CORRECTNESS_PROMPT = """Given the following query, context, model response, and the correct answer, rate how correct the model's response is.

**Scoring Rubric (1-10):**
- 1-2: Completely incorrect — wrong answer with wrong reasoning
- 3-4: Mostly incorrect — wrong answer, but reasoning shows partial understanding
- 5-6: Partially correct — right answer but weak/incorrect reasoning, OR wrong answer with sound reasoning
- 7-8: Mostly correct — right answer with reasonable explanation
- 9-10: Fully correct — right answer with accurate, well-supported explanation

**Query:**
{query}

**Context Provided to Model:**
{context}

**Model's Full Response:**
{response}

**Correct Answer:** {correct_answer}
**Model's Answer:** {model_answer}

Respond with ONLY a JSON object in this exact format:
{{"score": <integer 1-10>, "reasoning": "<one to two sentences explaining your score>"}}"""

RESPONSE_COMPLETENESS_PROMPT = """Given the following query and model response, rate how complete the response is in addressing the question.

**Scoring Rubric (1-10):**
- 1-2: No meaningful response — empty, garbled, or completely off-topic
- 3-4: Minimal response — provides an answer letter but no useful explanation
- 5-6: Partial response — provides answer with brief explanation but lacks depth
- 7-8: Good response — provides answer with clear, relevant explanation
- 9-10: Excellent response — provides answer with thorough, well-reasoned explanation that demonstrates understanding

**Query:**
{query}

**Model's Full Response:**
{response}

Respond with ONLY a JSON object in this exact format:
{{"score": <integer 1-10>, "reasoning": "<one to two sentences explaining your score>"}}"""

print("✓ Judge prompts defined (3 tasks: relevance, correctness, completeness)")

In [ ]:
def call_gpt41_judge(system_prompt, user_prompt, max_retries=3, retry_delay=5):
    """Call GPT-4.1 as judge and parse the JSON response.
    Returns {'score': int, 'reasoning': str}, or {'score': -1, 'reasoning': '<error>'} on failure."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4.1",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.1,
                max_tokens=200,
                response_format={"type": "json_object"},
            )
            content = response.choices[0].message.content.strip()
            result = json.loads(content)
            score = int(result.get('score', -1))
            if 1 <= score <= 10:
                return {'score': score, 'reasoning': result.get('reasoning', '')}
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
                continue
            return {'score': score, 'reasoning': f"Score out of range: {result}"}
        except json.JSONDecodeError as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
                continue
            return {'score': -1, 'reasoning': f"JSON parse error: {str(e)}"}
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
                continue
            return {'score': -1, 'reasoning': f"API error: {str(e)}"}
    return {'score': -1, 'reasoning': 'Max retries exceeded'}


def evaluate_single_question(question_result, method_name):
    """Run all 3 judge tasks on a single 23i question result."""
    query = question_result.get('query', '')
    context = question_result.get('context', '')
    model_answer = question_result.get('model_answer', '')
    correct_answer = question_result.get('correct_answer', '')
    full_response = question_result.get('full_response', '')
    explanation = question_result.get('explanation', '')

    response_text = full_response if full_response else f"Answer: {model_answer} - {explanation}"

    # Task 1: Context Relevance (skipped for no_context)
    if method_name == 'no_context' or not context:
        relevance = {'score': -1, 'reasoning': 'N/A — no context provided for this method'}
    else:
        ctx = context[:3000] + '...' if len(context) > 3000 else context
        relevance = call_gpt41_judge(
            JUDGE_SYSTEM_PROMPT,
            CONTEXT_RELEVANCE_PROMPT.format(query=query, context=ctx),
        )

    # Task 2: Response Correctness
    ctx_for_corr = (context[:2000] + '...') if len(context) > 2000 else (context if context else 'No context provided')
    correctness = call_gpt41_judge(
        JUDGE_SYSTEM_PROMPT,
        RESPONSE_CORRECTNESS_PROMPT.format(
            query=query, context=ctx_for_corr, response=response_text,
            correct_answer=correct_answer, model_answer=model_answer,
        ),
    )

    # Task 3: Response Completeness
    completeness = call_gpt41_judge(
        JUDGE_SYSTEM_PROMPT,
        RESPONSE_COMPLETENESS_PROMPT.format(query=query, response=response_text),
    )

    return {
        'query': query,
        'model_answer': model_answer,
        'correct_answer': correct_answer,
        'is_correct': question_result.get('is_correct', False),
        'relevance_score': relevance['score'],
        'relevance_reasoning': relevance['reasoning'],
        'correctness_score': correctness['score'],
        'correctness_reasoning': correctness['reasoning'],
        'completeness_score': completeness['score'],
        'completeness_reasoning': completeness['reasoning'],
    }

print("✓ Judge functions defined (call_gpt41_judge + evaluate_single_question)")

## Run the parallel judge evaluation

For each (model, method), submit all questions to a thread pool of 8 workers.
After every model finishes, the partial results are saved to disk so a crash doesn't lose progress.

In [ ]:
# Build evaluation plan and report scope
eval_plan = []
for model in MODELS_TO_JUDGE:
    for method in METHODS:
        n = len(all_results[model][method]['detailed_results'])
        eval_plan.append({'model': model, 'method': method, 'n_questions': n})

total_questions = sum(p['n_questions'] for p in eval_plan)
# 3 judge calls per question, minus relevance calls for no_context
no_context_questions = sum(p['n_questions'] for p in eval_plan if p['method'] == 'no_context')
total_api_calls = total_questions * 3 - no_context_questions

print(f"Evaluation plan:")
print(f"  {len(MODELS_TO_JUDGE)} models × {len(METHODS)} methods = {len(eval_plan)} combinations")
print(f"  Total questions to judge: {total_questions}")
print(f"  Estimated GPT-4.1 calls:  {total_api_calls}")
print(f"
Breakdown:")
for p in eval_plan:
    print(f"  {p['model']:35s}  {p['method']:<12s}  {p['n_questions']} questions")

In [ ]:
MAX_WORKERS = 8
SKIPPED_ANSWERS = {'ERROR', 'INVALID'}

def _judge_one(idx, q_result, method):
    """Worker function for the thread pool. Returns (idx, eval_dict)."""
    if q_result.get('model_answer') in SKIPPED_ANSWERS:
        return idx, {
            'query': q_result.get('query', ''),
            'model_answer': q_result.get('model_answer', ''),
            'correct_answer': q_result.get('correct_answer', ''),
            'is_correct': False,
            'relevance_score': -1, 'relevance_reasoning': 'Skipped — model returned ERROR/INVALID',
            'correctness_score': -1, 'correctness_reasoning': 'Skipped — model returned ERROR/INVALID',
            'completeness_score': -1, 'completeness_reasoning': 'Skipped — model returned ERROR/INVALID',
        }
    return idx, evaluate_single_question(q_result, method)


judge_results = {}  # judge_results[model][method] = [list of per-question dicts, in original order]
judge_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
intermediate_path = f'judge_results_23i_intermediate_{judge_timestamp}.json'

start_time = time.time()
total_evaluated = 0

for model in MODELS_TO_JUDGE:
    judge_results[model] = {}
    print(f"\n{'='*80}")
    print(f"Model: {model}")
    print(f"{'='*80}")

    for method in METHODS:
        detailed = all_results[model][method]['detailed_results']
        out_list = [None] * len(detailed)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(_judge_one, i, q, method) for i, q in enumerate(detailed)]
            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=f"  {method} ({len(detailed)} q, {MAX_WORKERS} workers)"):
                i, res = fut.result()
                out_list[i] = res
                total_evaluated += 1

        judge_results[model][method] = out_list

        # Per-method summary
        rel = [r['relevance_score'] for r in out_list if r['relevance_score'] > 0]
        cor = [r['correctness_score'] for r in out_list if r['correctness_score'] > 0]
        com = [r['completeness_score'] for r in out_list if r['completeness_score'] > 0]
        print(f"    relevance:    {(np.mean(rel) if rel else float('nan')):.2f}  ({len(rel)} scored)")
        print(f"    correctness:  {(np.mean(cor) if cor else float('nan')):.2f}  ({len(cor)} scored)")
        print(f"    completeness: {(np.mean(com) if com else float('nan')):.2f}  ({len(com)} scored)")

    # Save after each model (crash safety)
    with open(intermediate_path, 'w') as f:
        json.dump(judge_results, f, indent=2)

elapsed = time.time() - start_time
print(f"\n{'='*80}")
print(f"Judging complete — {total_evaluated} questions in {elapsed/60:.1f} min")
print(f"Intermediate saves at: {intermediate_path}")
print(f"{'='*80}")

In [ ]:
# Final save with full metadata wrapper
final_save_path = f'judge_results_23i_final_{judge_timestamp}.json'

output = {
    'metadata': {
        'source_file': RESULTS_FILE,
        'models_judged': MODELS_TO_JUDGE,
        'models_skipped': SKIPPED,
        'methods': METHODS,
        'judge_model': 'gpt-4.1',
        'max_workers': MAX_WORKERS,
        'timestamp': judge_timestamp,
        'total_questions_judged': total_evaluated,
        'elapsed_minutes': round(elapsed / 60, 1),
    },
    'results': judge_results,
}

with open(final_save_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"✓ Final judge results saved: {final_save_path}")
print(f"  Size: {os.path.getsize(final_save_path)/(1024*1024):.2f} MB")

## Aggregate to a summary DataFrame + CSV

In [ ]:
rows = []
for model in MODELS_TO_JUDGE:
    for method in METHODS:
        records = judge_results[model][method]
        rel = [r['relevance_score'] for r in records if r['relevance_score'] > 0]
        cor = [r['correctness_score'] for r in records if r['correctness_score'] > 0]
        com = [r['completeness_score'] for r in records if r['completeness_score'] > 0]
        rows.append({
            'model': model,
            'method': method,
            'n_questions': len(records),
            'relevance_mean': np.mean(rel) if rel else np.nan,
            'relevance_n': len(rel),
            'correctness_mean': np.mean(cor) if cor else np.nan,
            'correctness_n': len(cor),
            'completeness_mean': np.mean(com) if com else np.nan,
            'completeness_n': len(com),
            'mcq_accuracy': all_results[model][method].get('accuracy', np.nan),
        })

summary_df = pd.DataFrame(rows)
summary_csv = f'judge_summary_23i_{judge_timestamp}.csv'
summary_df.to_csv(summary_csv, index=False)
print(f"✓ Summary CSV: {summary_csv}")
summary_df

## Visualizations

**Plot 1:** grouped bar chart per task — 14 models on x-axis, 2 bars per model (no_context vs mmr).
**Plot 2:** heatmap per method — models (rows) × tasks (cols).

In [ ]:
# Plot 1: Grouped bar chart per task (3 figures)
TASKS = [
    ('relevance_mean', 'Context Relevance'),
    ('correctness_mean', 'Response Correctness'),
    ('completeness_mean', 'Response Completeness'),
]
METHOD_COLORS = {'no_context': '#FF9800', 'mmr': '#9C27B0'}

for task_col, task_title in TASKS:
    fig, ax = plt.subplots(figsize=(max(14, len(MODELS_TO_JUDGE) * 1.0), 6))
    x = np.arange(len(MODELS_TO_JUDGE))
    width = 0.4

    for i, method in enumerate(METHODS):
        sub = summary_df[summary_df['method'] == method].set_index('model').reindex(MODELS_TO_JUDGE)
        vals = sub[task_col].fillna(0).values
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=method,
                      color=METHOD_COLORS.get(method, None), alpha=0.85, edgecolor='black')
        for bar, v in zip(bars, vals):
            if not np.isnan(v) and v > 0:
                ax.text(bar.get_x() + bar.get_width()/2., v + 0.1, f'{v:.1f}',
                        ha='center', va='bottom', fontsize=7, fontweight='bold')

    ax.set_xlabel('Model', fontsize=11, fontweight='bold')
    ax.set_ylabel(f'{task_title} (mean, 1-10)', fontsize=11, fontweight='bold')
    ax.set_title(f'Mrigi 30a — {task_title}: no_context vs mmr (k=15)', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(MODELS_TO_JUDGE, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 10.5)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.legend(fontsize=10)
    plt.tight_layout()
    out = f'judge_23i_bars_{task_col}_{judge_timestamp}.svg'
    fig.savefig(out, format='svg', bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {out}")

In [ ]:
# Plot 2: Heatmap per method (models x tasks)
task_cols = ['relevance_mean', 'correctness_mean', 'completeness_mean']
task_labels = ['Relevance', 'Correctness', 'Completeness']

for method in METHODS:
    sub = summary_df[summary_df['method'] == method].set_index('model').reindex(MODELS_TO_JUDGE)
    data = sub[task_cols].values.astype(float)

    fig, ax = plt.subplots(figsize=(7, max(6, len(MODELS_TO_JUDGE) * 0.45)))
    im = ax.imshow(data, aspect='auto', cmap='YlGnBu', vmin=0, vmax=10)

    ax.set_xticks(range(len(task_labels)))
    ax.set_xticklabels(task_labels, fontsize=10)
    ax.set_yticks(range(len(MODELS_TO_JUDGE)))
    ax.set_yticklabels(MODELS_TO_JUDGE, fontsize=9)
    ax.set_title(f'Mrigi 30a — {method} (k=15): Models × Judge Tasks', fontsize=12, fontweight='bold')

    # Annotate cells
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            txt = f'{val:.1f}' if not np.isnan(val) else 'N/A'
            color = 'white' if (not np.isnan(val) and val > 6.5) else 'black'
            ax.text(j, i, txt, ha='center', va='center', fontsize=9, color=color, fontweight='bold')

    fig.colorbar(im, ax=ax, label='Mean score (1-10)')
    plt.tight_layout()
    out = f'judge_23i_heatmap_{method}_{judge_timestamp}.svg'
    fig.savefig(out, format='svg', bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {out}")

In [ ]:
# Final summary printout
print("="*80)
print("Mrigi 30a — LLM-as-Judge Evaluation Complete")
print("="*80)
print(f"Source file:        {RESULTS_FILE}")
print(f"Final results:      {final_save_path}")
print(f"Summary CSV:        {summary_csv}")
print(f"Models judged:      {len(MODELS_TO_JUDGE)}")
print(f"Models skipped:     {len(SKIPPED)}")
print(f"Total questions:    {total_evaluated}")
print(f"Elapsed time:       {elapsed/60:.1f} min")
print(f"Worker threads:     {MAX_WORKERS}")
print("="*80)